# Functional SlowHeat — benchmark Split MNIST class-incremental

Este notebook executa cinco tarefas sequenciais (`0/1`, `2/3`, ..., `8/9`) com um único MLP. A inferência não recebe task ID, classes futuras não participam da loss e todos os métodos usam a mesma inicialização, dados e sequência de minibatches.

O objetivo é medir simultaneamente **retenção** e **plasticidade**. Uma queda de forgetting acompanhada por queda maior de acurácia não é uma melhoria global.

## Preparação

Na raiz do repositório, instale uma vez as dependências: `python -m pip install -e '.[research]'`. Na primeira execução, o torchvision baixará o MNIST para `data/`.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from experiments.split_mnist import (
    SplitMNISTConfig,
    load_split_mnist,
    run_split_mnist,
)

plt.style.use('seaborn-v0_8-whitegrid')
print('PyTorch:', torch.__version__)
print('Dispositivo disponível:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Raiz:', ROOT)

## Parâmetros principais

Os valores abaixo formam um teste CPU razoavelmente rápido. Para um resultado mais forte, use todas as amostras (`train_per_class=None`, `test_per_class=None`), 5–10 épocas e várias seeds. O controle adaptativo usa apenas o split de validação.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'results' / 'split_mnist_notebook'

config = SplitMNISTConfig(
    seed=42,
    hidden_dims=(256, 128),
    batch_size=128,
    epochs_per_task=2,
    train_per_class=1_000,       # None usa todo o treino disponível
    validation_per_class=200,
    test_per_class=500,          # None usa todo o teste
    learning_rate=1e-3,
    weight_decay=1e-4,
    slow_strength=3.0,           # força máxima de proteção
    plasticity_budget=0.25,      # no mínimo 25% dos neurônios livres
    optimizer_state_policy='follow_update',
    adaptive_target_accuracy=0.90,
    adaptive_rate=0.20,
    adaptive_minimum=0.10,
    adaptive_maximum=0.80,
    methods=(
        'vanilla',
        'slowheat',
        'slowheat_adaptive',
        'slowheat_native_state',
        'slowheat_unidirectional',
    ),
    device=DEVICE,
)
config

In [ ]:
tasks = load_split_mnist(config, data_dir=DATA_DIR, download=True)
pd.DataFrame([
    {
        'task': index + 1,
        'classes': str(task.classes),
        'train': len(task.train_y),
        'validation': len(task.validation_y),
        'test': len(task.test_y),
    }
    for index, task in enumerate(tasks)
])

## Comparação principal

- `vanilla`: AdamW sem proteção.
- `slowheat`: método completo, orçamento fixo.
- `slowheat_adaptive`: ajusta o orçamento usando aquisição de validação.
- `slowheat_native_state`: não mascara os momentos do AdamW.
- `slowheat_unidirectional`: protege linhas, mas não as colunas da camada seguinte.

In [ ]:
results = run_split_mnist(config, tasks, output_dir=OUTPUT_DIR)

summary = pd.DataFrame([
    {
        'method': method,
        'final_accuracy': result['metrics']['final_average_accuracy'],
        'forgetting': result['metrics']['average_forgetting'],
        'BWT': result['metrics']['backward_transfer'],
        'FWT': result['metrics']['forward_transfer'],
        'seconds': result['elapsed_seconds'],
    }
    for method, result in results.items()
]).sort_values('final_accuracy', ascending=False)
summary.style.format({
    'final_accuracy': '{:.4f}', 'forgetting': '{:.4f}',
    'BWT': '{:.4f}', 'FWT': '{:.4f}', 'seconds': '{:.1f}',
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
stages = np.arange(1, config.task_count + 1)
for method, result in results.items():
    axes[0].plot(stages, result['stage_average_accuracy'], marker='o', label=method)
    axes[1].plot(stages, result['stage_average_forgetting'], marker='o', label=method)
axes[0].set(title='Acurácia média ao longo do stream', xlabel='Tarefas aprendidas', ylabel='Acurácia média', xticks=stages, ylim=(0, 1))
axes[1].set(title='Forgetting acumulado', xlabel='Tarefas aprendidas', ylabel='Forgetting médio', xticks=stages, ylim=(0, 1))
axes[1].legend(loc='best', fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
method_names = list(results)
fig, axes = plt.subplots(1, len(method_names), figsize=(4 * len(method_names), 3.8), squeeze=False)
for axis, method in zip(axes[0], method_names):
    matrix = np.array([[np.nan if value is None else value for value in row] for row in results[method]['accuracy_matrix']])
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    for row in range(matrix.shape[0]):
        for col in range(row + 1):
            axis.text(col, row, f'{matrix[row, col]:.2f}', ha='center', va='center', color='white' if matrix[row, col] < 0.55 else 'black', fontsize=8)
    axis.set(title=method, xlabel='Task avaliada', ylabel='Após task', xticks=range(config.task_count), yticks=range(config.task_count))
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.8, label='Acurácia')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for method in ('slowheat', 'slowheat_adaptive'):
    if method not in results:
        continue
    axes[0].plot(stages, results[method]['validation_acquisition'], marker='o', label=method)
    history = results[method]['capacity_history']
    plastic = [np.mean([layer['plastic_fraction'] for layer in stage]) for stage in history]
    axes[1].plot(stages[:len(plastic)], plastic, marker='o', label=method)
axes[0].axhline(config.adaptive_target_accuracy, color='black', linestyle='--', label='meta adaptativa')
axes[0].set(title='Aquisição da tarefa atual (validação)', xlabel='Task', ylabel='Acurácia', xticks=stages, ylim=(0, 1))
axes[1].set(title='Capacidade plástica realizada', xlabel='Task', ylabel='Fração plástica média', xticks=stages, ylim=(0, 1))
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## Sweep dos parâmetros mais importantes

Este sweep usa uma configuração menor para comparar rapidamente força de proteção e capacidade livre. No gráfico de Pareto, o melhor sentido é **direita e baixo**: mais acurácia, menos forgetting.

In [ ]:
RUN_SWEEP = True
SWEEP_STRENGTHS = [1.0, 3.0, 6.0]
SWEEP_BUDGETS = [0.25, 0.50]
SWEEP_EPOCHS = 1
SWEEP_TRAIN_PER_CLASS = 500

sweep_rows = []
if RUN_SWEEP:
    for strength in SWEEP_STRENGTHS:
        for budget in SWEEP_BUDGETS:
            sweep_config = replace(
                config,
                methods=('slowheat',),
                slow_strength=strength,
                plasticity_budget=budget,
                epochs_per_task=SWEEP_EPOCHS,
                train_per_class=SWEEP_TRAIN_PER_CLASS,
            )
            sweep_tasks = load_split_mnist(sweep_config, data_dir=DATA_DIR, download=False)
            run = run_split_mnist(sweep_config, sweep_tasks)['slowheat']
            sweep_rows.append({
                'slow_strength': strength,
                'plasticity_budget': budget,
                'final_accuracy': run['metrics']['final_average_accuracy'],
                'forgetting': run['metrics']['average_forgetting'],
                'BWT': run['metrics']['backward_transfer'],
            })
    sweep = pd.DataFrame(sweep_rows)
    sweep.to_csv(OUTPUT_DIR / 'parameter_sweep.csv', index=False)
else:
    sweep = pd.DataFrame(columns=['slow_strength', 'plasticity_budget', 'final_accuracy', 'forgetting', 'BWT'])
sweep

In [ ]:
if not sweep.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for budget, group in sweep.groupby('plasticity_budget'):
        ordered = group.sort_values('slow_strength')
        axes[0].plot(ordered['slow_strength'], ordered['final_accuracy'], marker='o', label=f'budget={budget}')
        axes[1].plot(ordered['slow_strength'], ordered['forgetting'], marker='o', label=f'budget={budget}')
    axes[0].set(title='Sensibilidade da acurácia', xlabel='slow_strength', ylabel='Acurácia final', ylim=(0, 1))
    axes[1].set(title='Sensibilidade do forgetting', xlabel='slow_strength', ylabel='Forgetting', ylim=(0, 1))
    axes[0].legend(); axes[1].legend()
    for _, row in sweep.iterrows():
        axes[2].scatter(row['final_accuracy'], row['forgetting'], s=60)
        axes[2].annotate(f"β={row['slow_strength']}, p={row['plasticity_budget']}", (row['final_accuracy'], row['forgetting']), xytext=(4, 4), textcoords='offset points', fontsize=8)
    axes[2].set(title='Fronteira estabilidade–plasticidade', xlabel='Acurácia final → melhor', ylabel='Forgetting → melhor')
    fig.tight_layout()
    plt.show()

## Leitura automática do resultado

A célula abaixo compara o método completo com vanilla. Para afirmar melhora global, procure acurácia igual ou maior **e** forgetting menor, com repetição em várias seeds.

In [ ]:
vanilla = results['vanilla']['metrics']
candidate = results['slowheat']['metrics']
delta_accuracy = candidate['final_average_accuracy'] - vanilla['final_average_accuracy']
delta_forgetting = candidate['average_forgetting'] - vanilla['average_forgetting']
print(f'Delta de acurácia final: {delta_accuracy:+.4f}')
print(f'Delta de forgetting:      {delta_forgetting:+.4f}  (negativo é melhor)')
if delta_accuracy >= 0 and delta_forgetting <= 0:
    print('Resultado preliminar dominante: melhorou ou preservou ambas as métricas.')
elif delta_forgetting < 0:
    print('Trade-off: houve maior retenção, mas verifique o custo de plasticidade na acurácia.')
else:
    print('Nesta configuração não houve evidência de melhora sobre vanilla.')
print('Artefatos salvos em:', OUTPUT_DIR)

## Execução mais rigorosa

Para uma avaliação científica: use `train_per_class=None`, `test_per_class=None`, 5–10 épocas, seeds predefinidas e reporte média/intervalo pareado. Split MNIST é um benchmark de depuração; o passo seguinte deve ser Split CIFAR-100/TinyImageNet com baselines especializados.